# Physical AI Workshop — End-to-End Walkthrough

This notebook runs the same pipeline the `scripts/` CLIs run, but with inline
commentary so attendees can see what each stage produces.

Flow:
1. **Train** a Diffusion Policy on PushT with `lerobot-train`.
2. **Evaluate** the checkpoint and log rollout videos + Weave traces.
3. **Compare** the evaluation against a baseline (run ID or JSON).
4. **Promote** the checkpoint as a W&B Model artifact.
5. **Weave Evaluation** over the rollouts with four scorers.


## 0. Environment check

In [ ]:
import os, sys, pathlib
ROOT = pathlib.Path.cwd().resolve()
if (ROOT / 'src').exists() is False and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
print('repo root:', ROOT)

import torch
print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available())

os.environ.setdefault('WANDB_PROJECT', 'physical-ai-workshop')

## 1. Train Diffusion Policy (smoke)

200 steps runs in ~3 minutes on a single H100. For the real workshop
candidate, bump `--steps` to 10000 and drop `--eval-freq 0`.

In [ ]:
!python scripts/01_train_diffusion_pusht.py \
    --output-dir workspace/runs/nb-smoke \
    --run-name nb-smoke \
    --steps 200 --eval-freq 0 --save-freq 200 --batch-size 32

## 2. Evaluate the checkpoint

This produces `eval_summary.json`, a W&B run with `eval/rollouts` +
`eval/failures` video tables, and Weave traces per episode.

In [ ]:
!python scripts/03_evaluate.py \
    --checkpoint workspace/runs/nb-smoke/checkpoints/last/pretrained_model \
    --output-dir workspace/runs/nb-smoke/eval \
    --num-episodes 10 --record-every 2 --tag nb-smoke \
    --weave-project physical-ai-workshop

In [ ]:
import json
summary = json.loads(pathlib.Path('workspace/runs/nb-smoke/eval/eval_summary.json').read_text())
print('metrics:', json.dumps(summary['metrics'], indent=2))
print('episodes:', len(summary['episodes']))
print('first failure modes:', [e['failure_mode'] for e in summary['episodes'][:5]])

### Inspect one rollout video inline

In [ ]:
from IPython.display import Video
videos = sorted(pathlib.Path('workspace/runs/nb-smoke/eval/videos').glob('*.mp4'))
print('rendered:', [v.name for v in videos[:3]])
Video(str(videos[0])) if videos else 'no videos yet'

## 3. Compare two candidates

Compare the smoke against itself as a dry-run; in the live workshop point
`--right` at the baseline eval and keep this smoke as the candidate.

In [ ]:
!python scripts/04_compare.py \
    --left workspace/runs/nb-smoke/eval/eval_summary.json \
    --right workspace/runs/nb-smoke/eval/eval_summary.json \
    --left-label baseline --right-label candidate \
    --output workspace/reports/nb-compare.md
print(pathlib.Path('workspace/reports/nb-compare.md').read_text())

## 4. Promote the checkpoint as a W&B Model

Drop `--registry-collection` if the workshop entity does not have a
Registry collection yet.

In [ ]:
!python scripts/05_promote_to_registry.py \
    --checkpoint workspace/runs/nb-smoke/checkpoints/last/pretrained_model \
    --eval-summary workspace/runs/nb-smoke/eval/eval_summary.json \
    --artifact-name pusht-diffusion \
    --alias candidate

## 5. Weave Evaluation over the rollouts

Runs four scorers (`success`, `coverage`, `efficiency`, `failure_mode`)
against the stored rollouts. Requires network access to Weave.

In [ ]:
!python scripts/06_weave_evaluation.py \
    --eval-summary workspace/runs/nb-smoke/eval/eval_summary.json \
    --weave-project physical-ai-workshop \
    --dataset-name pusht-rollouts-nb

## 6. ART rollout critic (dry-run)

Builds an `art.Trajectory` per rollout where an LLM supervisor proposes a
recovery action. Dry-run mode uses a deterministic synthetic LLM so the
pipeline runs without a GPU or network.


In [ ]:
!python scripts/07_art_critic.py \
    --eval-summary workspace/runs/nb-smoke/eval/eval_summary.json \
    --output workspace/runs/nb-smoke/critic/summary.json

In [ ]:
import json, pathlib
critic = json.loads(pathlib.Path('workspace/runs/nb-smoke/critic/summary.json').read_text())
print('summary:', json.dumps(critic['summary'], indent=2))
print('example critique:')
print(json.dumps(critic['critiques'][0], indent=2))


## Where to go next

- Swap `scripts/01_train_diffusion_pusht.py` for `scripts/02_train_act_pusht.py`
  to train an ACT policy as the second candidate.
- Run `scripts/04_compare.py --left-run entity/project/<baseline>` against a
  live W&B run instead of a local JSON.
- Point `env_utils.make_pusht_env` at an Isaac Lab env to run the exact same
  pipeline on a different robot task.
